# Piu' teste: lo stesso calcolo, su fette diverse dello stesso spazio

*Anatomia di un LLM* · capitolo 5, «L'attenzione e il blocco» · nel libro: §5.2 «Sei teste invece di una» · script `03-05`, `scripts/03_05_multihead.py`

**Dove siamo.** Le teste: la stessa attenzione, fatta sei volte su sei fette dei numeri di ogni parola. Qui si controlla che non costino niente e che siano davvero sei conti separati.

Una testa sola vede una relazione alla volta. Ma in una frase ce ne sono molte
in corso insieme -- l'accordo del verbo, il referente del pronome, la parentesi
ancora aperta -- e una sola distribuzione di percentuali non puo' portarle
tutte. La soluzione non e' un meccanismo nuovo: e' lo stesso, ripetuto in
parallelo su porzioni disgiunte dello spazio.

Si esegue con `python3 scripts/03_05_multihead.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_05_multihead.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import torch

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.model import CausalSelfAttention
from llm_anatomy.utils import (conta_parametri, numero, prepara, tabella, titolo,
                               verifica)

prepara()

B = 2       # due frasi in parallelo
T = 16      # sedici parole ciascuna
C = 96      # novantasei numeri per parola
h = 6       # sei teste
d_head = C // h

cfg = AgoGPTConfig(d_model=C, n_head=h, block_size=T, vocab_size=93)
attn = CausalSelfAttention(cfg).eval()
x = torch.randn(B, T, C)

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


### Dividere non vuol dire aggiungere

> **Nel libro, §5.2 «Sei teste invece di una».** Come si seguono più relazioni insieme, senza allargare il modello?

**Cosa guardare.** Le due forme: 96 numeri riletti come 6 gruppi da 16, senza aggiungerne nessuno.

In [ ]:
titolo("Dividere non vuol dire aggiungere")
q = attn.wq(x)
# Il gesto e' tutto qui: gli stessi 96 numeri vengono riletti come 6 gruppi da
# 16. view() non sposta un byte, riscrive solo come interpretare la memoria;
# transpose porta le teste davanti, cosi' che il calcolo che segue le tratti
# come se fossero altrettante frasi indipendenti.
qh = q.view(B, T, h, d_head).transpose(1, 2)
print(f"  una parola entra come {C} numeri")
print(f"  esce dalla proiezione ancora come {C} numeri")
print(f"  che rileggiamo come {h} gruppi da {d_head}: {h} x {d_head} = {C}")
print("  nessun numero e' stato aggiunto e nessuno perso")
print(f"\n  forma prima:  {tuple(q.shape)}")
print(f"  forma dopo:   {tuple(qh.shape)}")


Dividere non vuol dire aggiungere
---------------------------------
  una parola entra come 96 numeri
  esce dalla proiezione ancora come 96 numeri
  che rileggiamo come 6 gruppi da 16: 6 x 16 = 96
  nessun numero e' stato aggiunto e nessuno perso

  forma prima:  (2, 16, 96)
  forma dopo:   (2, 6, 16, 16)


### Sei teste costano quanto una

**Cosa guardare.** La colonna dei parametri non si muove.

**Prova a cambiare:** aggiungi `96` all'elenco delle teste.

<details><summary>Che cosa succede</summary>

Novantasei teste da un numero ciascuna costano ancora 36.864 parametri: si possono fare, ma ogni testa ha un solo numero con cui confrontare.

</details>

In [ ]:
titolo("Sei teste costano quanto una")
# La sorpresa: le quattro matrici dell'attenzione restano C x C qualunque sia
# il numero di teste, perche' le teste si spartiscono C invece di moltiplicarlo.
# Aumentare h non e' una spesa: e' una scelta su come dividere lo stesso spazio.
righe = []
for teste in (1, 2, 4, 6, 12):
    c = AgoGPTConfig(d_model=C, n_head=teste, block_size=T, vocab_size=93)
    n = conta_parametri(CausalSelfAttention(c))
    righe.append([teste, C // teste, numero(n)])
tabella(["teste", "numeri per testa", "parametri"], righe)
print("  la colonna dei parametri non si muove")
print("  piu' teste = fette piu' sottili, non piu' modello")


Sei teste costano quanto una
----------------------------
teste  numeri per testa  parametri
-----  ----------------  ---------
1                    96     36.864
2                    48     36.864
4                    24     36.864
6                    16     36.864
12                    8     36.864
  la colonna dei parametri non si muove
  piu' teste = fette piu' sottili, non piu' modello


### Con una testa sola non è cambiato niente

*Questa parte non è stampata nel libro.*

**Cosa guardare.** Con una testa si torna esattamente all'attenzione del notebook 03-01.

In [ ]:
titolo("Con una testa sola non e' cambiato niente")
# Controprova: se il multi-head fosse un meccanismo diverso, con h=1 darebbe
# un risultato diverso dall'attenzione del primo script. Non lo da'.
c1 = AgoGPTConfig(d_model=C, n_head=1, block_size=T, vocab_size=93)
a1 = CausalSelfAttention(c1).eval()
with torch.no_grad():
    y1, _ = a1(x)
    q1, k1, v1 = a1.wq(x), a1.wk(x), a1.wv(x)
    s = (q1 @ k1.transpose(-2, -1)) / C ** 0.5
    m = torch.ones(T, T, dtype=torch.bool).tril()
    a_mano = a1.wo(s.masked_fill(~m, float("-inf")).softmax(-1) @ v1)
verifica(torch.allclose(y1, a_mano, atol=1e-5),
         "con una testa sola si torna all'attenzione scritta a mano")


Con una testa sola non e' cambiato niente
-----------------------------------------
  [verifica superata] con una testa sola si torna all'attenzione scritta a
                      mano


### Con sei teste sono sei conti separati

**Cosa guardare.** Sei attenzioni separate, riattaccate con `torch.cat`, danno lo stesso risultato del modulo.

In [ ]:
titolo("Con sei teste sono sei conti separati")
# Seconda controprova, piu' forte: si rifanno le sei attenzioni una per una,
# ciascuna sulla propria fetta, e si riconcatenano. Se il risultato coincide,
# allora "multi-head" e' davvero solo questo.
with torch.no_grad():
    y, _ = attn(x)
    q, k, v = (t(x).view(B, T, h, d_head).transpose(1, 2)
               for t in (attn.wq, attn.wk, attn.wv))
    m = torch.ones(T, T, dtype=torch.bool).tril()
    pezzi = []
    for i in range(h):
        s = (q[:, i] @ k[:, i].transpose(-2, -1)) / d_head ** 0.5
        pezzi.append(s.masked_fill(~m, float("-inf")).softmax(-1) @ v[:, i])
    ricomposta = attn.wo(torch.cat(pezzi, dim=-1))
verifica(torch.allclose(y, ricomposta, atol=1e-5),
         f"{h} attenzioni separate, riconcatenate, danno lo stesso risultato")
print("  e' la definizione di multi-head, verificata invece che dichiarata")


Con sei teste sono sei conti separati
-------------------------------------
  [verifica superata] 6 attenzioni separate, riconcatenate, danno lo stesso
                      risultato
  e' la definizione di multi-head, verificata invece che dichiarata


### Le teste guardano posti diversi

**Cosa guardare.** Le righe si somigliano tutte, intorno al sei per cento: a pesi casuali le teste non si sono ancora divise il lavoro.

In [ ]:
titolo("Le teste guardano posti diversi")
# L'ultima riga della matrice dei pesi: dove guarda l'ultima parola, secondo
# ciascuna testa. A pesi casuali le sei distribuzioni si somigliano; dopo
# l'addestramento divergono, ed e' quella divergenza il lavoro delle teste.
with torch.no_grad():
    _, pesi = attn(x, restituisci_pesi=True)
print("  ultima parola della prima frase, dove guarda secondo ogni testa:")
print("  (percentuale sulle prime otto posizioni)\n")
for i in range(h):
    riga = " ".join(f"{p:5.0%}" for p in pesi[0, i, -1, :8].tolist())
    print(f"  testa {i}  {riga}")


Le teste guardano posti diversi
-------------------------------
  ultima parola della prima frase, dove guarda secondo ogni testa:
  (percentuale sulle prime otto posizioni)

  testa 0     8%    6%    4%    4%    7%    8%    8%    6%
  testa 1     7%    6%    6%    7%    8%    6%    6%    8%
  testa 2     5%    5%   12%    9%    8%    8%    4%    6%
  testa 3     6%   11%    5%    7%    7%    6%    4%    4%
  testa 4     5%    5%    9%    6%    8%    6%    5%    8%
  testa 5     4%    4%    4%    6%    9%    4%    8%   10%
